# Modelo predictivo de riesgo de dengue por entidad federativa

Clasifica cada entidad federativa, para cada trimestre, en un nivel de riesgo de
dengue (**ALTO / MEDIO / BAJO**), usando como predictores únicamente información
de trimestres **anteriores** al que se quiere predecir (clima e incidencia
rezagados 1 y 2 trimestres). Esto permite que el modelo sea utilizable para
predecir un trimestre que todavía no ha ocurrido, sin depender de datos que en
ese momento no existirían.

**Entradas**:
- `FACT_CASOS_DENGUE.csv`, `FACT_POBLACION.csv`, `FACT_CLIMA.csv`
- `DIM_GEOGRAFIA.csv`, `DIM_TIEMPO.csv`

**Salida**: modelo Random Forest entrenado (`modelo_rf.pkl`) y métricas de
evaluación.


In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

fact_clima = pd.read_csv("HECHOS/FACT_CLIMA.csv")
fact_casos = pd.read_csv("HECHOS/FACT_CASOS_DENGUE.csv")
fact_pob = pd.read_csv("HECHOS/FACT_POBLACION.csv")
dim_geo = pd.read_csv("DIMENSIONES/DIM_GEOGRAFIA.csv")
dim_tiempo = pd.read_csv("DIMENSIONES/DIM_TIEMPO.csv")

dim_tiempo['fecha'] = pd.to_datetime(dim_tiempo['fecha'], dayfirst=True)
dim_tiempo['trimestre'] = dim_tiempo['fecha'].dt.to_period('Q')
dim_tiempo['anio'] = dim_tiempo['fecha'].dt.year

geo_to_entidad = dim_geo.set_index('id_geografia')['entidad'].to_dict()

## 1. Agregar cada tabla de hechos al grano estado-trimestre (o estado-año)

`FACT_CLIMA` y `FACT_CASOS_DENGUE` están a grano diario; `FACT_POBLACION` a
grano anual. Se agregan todas a nivel **entidad federativa**, ya que el clima
solo está disponible a ese nivel geográfico.

In [2]:
# --- Clima: diario-estado -> trimestre-estado ---
clima = fact_clima.merge(dim_tiempo[['id_tiempo', 'trimestre']], on='id_tiempo', how='left')
clima['entidad'] = clima['id_geografia'].map(geo_to_entidad)
clima_trim = clima.groupby(['entidad', 'trimestre']).agg(
    temp_max_prom=('temp_max', 'mean'),
    temp_min_prom=('temp_min', 'mean'),
    lluvia_total=('lluvia_acumulada', 'sum'),
    evapotrans_prom=('evapotranspiracion', 'mean')
).reset_index()

# --- Casos: diario-municipio -> trimestre-estado (solo estados con clima disponible) ---
casos = fact_casos.merge(dim_tiempo[['id_tiempo', 'trimestre']], on='id_tiempo', how='left')
casos['entidad'] = casos['id_geografia'].map(geo_to_entidad)
casos_trim = casos.groupby(['entidad', 'trimestre']).size().reset_index(name='casos')
estados_clima = set(clima_trim['entidad'].unique())
casos_trim = casos_trim[casos_trim['entidad'].isin(estados_clima)]

# --- Poblacion: municipio-sexo-edad-año -> estado-año ---
pob = fact_pob.merge(dim_tiempo[['id_tiempo', 'anio']].drop_duplicates(), on='id_tiempo', how='left')
pob['entidad'] = pob['id_geografia'].map(geo_to_entidad)
pob_anio = pob.groupby(['entidad', 'anio'])['poblacion'].sum().reset_index()

print("Clima trimestre-estado:", clima_trim.shape)
print("Casos trimestre-estado:", casos_trim.shape)
print("Poblacion año-estado:", pob_anio.shape)

Clima trimestre-estado: (754, 6)
Casos trimestre-estado: (614, 3)
Poblacion año-estado: (288, 3)


## 2. Unificar en una sola tabla base y calcular incidencia por 100,000 habitantes

In [3]:
df = clima_trim.merge(casos_trim, on=['entidad', 'trimestre'], how='left')
df['casos'] = df['casos'].fillna(0)  # trimestre sin casos reportados = 0 casos

df['anio'] = df['trimestre'].dt.year
df = df.merge(pob_anio, on=['entidad', 'anio'], how='left')
df['incidencia_100k'] = (df['casos'] / df['poblacion']) * 100000

df = df.sort_values(['entidad', 'trimestre']).reset_index(drop=True)
print("Filas base (estado-trimestre):", df.shape)
print("Nulos:", df.isnull().sum().sum())
df.head()

Filas base (estado-trimestre): (754, 10)
Nulos: 0


,entidad,trimestre,temp_max_prom,temp_min_prom,lluvia_total,evapotrans_prom,casos,anio,poblacion,incidencia_100k
0,AGUASCALIENTES,2020Q1,24.534066,9.548352,39.6,4.565165,0.0,2020,1456050,0.0
1,AGUASCALIENTES,2020Q2,29.809890,14.487912,82.9,6.641099,0.0,2020,1456050,0.0
2,AGUASCALIENTES,2020Q3,26.291304,14.521739,242.9,4.802391,0.0,2020,1456050,0.0
3,AGUASCALIENTES,2020Q4,25.284783,9.808696,5.0,4.614674,0.0,2020,1456050,0.0
4,AGUASCALIENTES,2021Q1,25.684444,8.573333,1.2,5.326222,0.0,2021,1472645,0.0


## 3. Definir los niveles de riesgo (variable objetivo)

Se usan **terciles** de la distribución histórica de incidencia para definir
ALTO / MEDIO / BAJO de forma relativa a la realidad observada (en vez de un
umbral epidemiológico fijo, que puede no ser representativo de este dataset).

In [4]:
q1, q2 = df['incidencia_100k'].quantile([1/3, 2/3])
print(f"Umbrales -> BAJO: <{q1:.3f} | MEDIO: {q1:.3f}-{q2:.3f} | ALTO: >{q2:.3f} (casos por 100k)")

def clasificar_riesgo(x):
    if x <= q1:
        return 'BAJO'
    elif x <= q2:
        return 'MEDIO'
    return 'ALTO'

df['riesgo'] = df['incidencia_100k'].apply(clasificar_riesgo)
print(df['riesgo'].value_counts())

Umbrales -> BAJO: <0.642 | MEDIO: 0.642-16.442 | ALTO: >16.442 (casos por 100k)
riesgo
ALTO     252
BAJO     251
MEDIO    251
Name: count, dtype: int64


## 4. Variables rezagadas (lag) — el corazón del diseño predictivo

Para que el modelo sea utilizable en una predicción real hacia el futuro,
**ninguna variable puede provenir del propio trimestre que se quiere predecir**.
Se usan solo los rezagos t-1 y t-2, su tendencia, y la estacionalidad
(codificada de forma cíclica con seno/coseno).

In [5]:
df = df.sort_values(['entidad', 'trimestre']).reset_index(drop=True)

lag_cols = ['temp_max_prom', 'temp_min_prom', 'lluvia_total', 'evapotrans_prom', 'casos', 'incidencia_100k']
for col in lag_cols:
    df[f'{col}_lag1'] = df.groupby('entidad')[col].shift(1)
    df[f'{col}_lag2'] = df.groupby('entidad')[col].shift(2)

df['tendencia_incidencia'] = df['incidencia_100k_lag1'] - df['incidencia_100k_lag2']

df['num_trimestre'] = df['trimestre'].dt.quarter
df['trimestre_sin'] = np.sin(2 * np.pi * df['num_trimestre'] / 4)
df['trimestre_cos'] = np.cos(2 * np.pi * df['num_trimestre'] / 4)

# Eliminar las primeras 2 observaciones de cada estado (no tienen lag completo)
cols_lag = [c for c in df.columns if 'lag' in c] + ['tendencia_incidencia']
df_modelo = df.dropna(subset=cols_lag).copy()

print("Filas disponibles para modelar:", df_modelo.shape)
print("Rango de trimestres:", df_modelo['trimestre'].min(), "-", df_modelo['trimestre'].max())

Filas disponibles para modelar: (696, 27)
Rango de trimestres: 2020Q3 - 2026Q2


## 5. Entrenamiento con partición temporal

Se entrena con todo lo anterior a 2025-Q3 y se prueba con los cuatro trimestres
más recientes — **nunca** una partición aleatoria, porque eso permitiría que el
modelo "viera" información futura durante el entrenamiento.

In [6]:
features = ['temp_max_prom_lag1', 'temp_min_prom_lag1', 'lluvia_total_lag1', 'evapotrans_prom_lag1',
            'casos_lag1', 'incidencia_100k_lag1',
            'temp_max_prom_lag2', 'temp_min_prom_lag2', 'lluvia_total_lag2', 'evapotrans_prom_lag2',
            'casos_lag2', 'incidencia_100k_lag2',
            'tendencia_incidencia', 'trimestre_sin', 'trimestre_cos']

X = df_modelo[features]
y = df_modelo['riesgo']

corte = pd.Period('2025Q3', freq='Q')
train_mask = df_modelo['trimestre'] < corte
test_mask = df_modelo['trimestre'] >= corte

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print("Entrenamiento:", X_train.shape, "| Prueba:", X_test.shape)
print("Trimestres de prueba:", sorted(df_modelo[test_mask]['trimestre'].unique().astype(str)))

Entrenamiento: (580, 15) | Prueba: (116, 15)
Trimestres de prueba: ['2025Q3', '2025Q4', '2026Q1', '2026Q2']


In [7]:
modelo = RandomForestClassifier(
    n_estimators=300, max_depth=6, min_samples_leaf=5,
    class_weight='balanced', random_state=42
)
modelo.fit(X_train, y_train)

y_pred = modelo.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred))
print("\nMatriz de confusión (orden ALTO/MEDIO/BAJO):")
print(confusion_matrix(y_test, y_pred, labels=['ALTO', 'MEDIO', 'BAJO']))

Accuracy: 0.8448275862068966

Reporte de clasificación:
              precision    recall  f1-score   support

        ALTO       0.88      0.96      0.92        55
        BAJO       0.86      0.63      0.73        19
       MEDIO       0.79      0.79      0.79        42

    accuracy                           0.84       116
   macro avg       0.84      0.79      0.81       116
weighted avg       0.84      0.84      0.84       116


Matriz de confusión (orden ALTO/MEDIO/BAJO):
[[53  2  0]
 [ 7 33  2]
 [ 0  7 12]]


## 6. Importancia de variables

In [8]:
importancia = pd.Series(modelo.feature_importances_, index=features).sort_values(ascending=False)
print(importancia)

incidencia_100k_lag1    0.233849
casos_lag1              0.207404
incidencia_100k_lag2    0.107078
casos_lag2              0.104335
tendencia_incidencia    0.086527
temp_min_prom_lag1      0.046802
temp_max_prom_lag1      0.038512
evapotrans_prom_lag1    0.030135
temp_min_prom_lag2      0.028319
trimestre_sin           0.023825
temp_max_prom_lag2      0.022334
lluvia_total_lag2       0.020976
lluvia_total_lag1       0.020751
evapotrans_prom_lag2    0.016103
trimestre_cos           0.013049
dtype: float64


## 7. Guardar el modelo entrenado

In [9]:
import pickle

with open('modelo_rf.pkl', 'wb') as f:
    pickle.dump({'modelo': modelo, 'features': features, 'umbral_bajo': q1, 'umbral_alto': q2}, f)

df_modelo.to_pickle('df_modelo_final.pkl')
print("Guardado: modelo_rf.pkl, df_modelo_final.pkl")

Guardado: modelo_rf.pkl, df_modelo_final.pkl


## 8. Probar el modelo con datos nuevos

Dos formas de poner a prueba el modelo ya entrenado:

1. **Predicción hacia adelante**: armar a mano una fila con los rezagos reales de
   los dos trimestres más recientes disponibles, para predecir un trimestre que
   aún no ha ocurrido.
2. **Validación contra un caso ya conocido**: tomar una fila del propio conjunto
   de prueba (`df_modelo_final.pkl`) y comparar la predicción contra el riesgo
   real observado.

Recordatorio importante: el modelo predice el riesgo del trimestre **t**
usando información de **t-1** y **t-2** — nunca variables del propio trimestre
que se quiere predecir.

In [10]:
import pickle
import pandas as pd

with open('modelo_rf.pkl', 'rb') as f:
    d = pickle.load(f)

modelo = d['modelo']
features = d['features']
umbral_bajo = d['umbral_bajo']
umbral_alto = d['umbral_alto']

print("Variables que espera el modelo:")
print(features)

Variables que espera el modelo:
['temp_max_prom_lag1', 'temp_min_prom_lag1', 'lluvia_total_lag1', 'evapotrans_prom_lag1', 'casos_lag1', 'incidencia_100k_lag1', 'temp_max_prom_lag2', 'temp_min_prom_lag2', 'lluvia_total_lag2', 'evapotrans_prom_lag2', 'casos_lag2', 'incidencia_100k_lag2', 'tendencia_incidencia', 'trimestre_sin', 'trimestre_cos']


### 8.1 Predicción automática con los datos más recientes disponibles

En vez de armar los rezagos a mano, se toman automáticamente, para **cada
entidad**, los dos trimestres más recientes que ya tengamos completos en
`df_modelo` (que se vuelven lag1 y lag2), y se predice el riesgo del
**trimestre inmediato siguiente** — el primero que todavía no ha ocurrido en
los datos.

In [11]:
# Trimestre mas reciente disponible por entidad (normalmente el mismo para todas,
# pero se calcula por si alguna entidad tuviera un corte distinto)
ultimo_trimestre_global = df_modelo['trimestre'].max()
trimestre_a_predecir = ultimo_trimestre_global + 1

print("Ultimo trimestre con datos:", ultimo_trimestre_global)
print("Trimestre a predecir:", trimestre_a_predecir)

filas_prediccion = []
for entidad, grupo in df_modelo.groupby('entidad'):
    grupo = grupo.sort_values('trimestre')
    ultima_fila = grupo.iloc[-1]  # el trimestre mas reciente se convierte en lag1

    # Los propios valores "actuales" de la ultima fila pasan a ser el lag1 del trimestre a predecir
    fila = {
        'entidad': entidad,
        'trimestre_predicho': str(trimestre_a_predecir),

        'temp_max_prom_lag1': ultima_fila['temp_max_prom'],
        'temp_min_prom_lag1': ultima_fila['temp_min_prom'],
        'lluvia_total_lag1': ultima_fila['lluvia_total'],
        'evapotrans_prom_lag1': ultima_fila['evapotrans_prom'],
        'casos_lag1': ultima_fila['casos'],
        'incidencia_100k_lag1': ultima_fila['incidencia_100k'],

        'temp_max_prom_lag2': ultima_fila['temp_max_prom_lag1'],
        'temp_min_prom_lag2': ultima_fila['temp_min_prom_lag1'],
        'lluvia_total_lag2': ultima_fila['lluvia_total_lag1'],
        'evapotrans_prom_lag2': ultima_fila['evapotrans_prom_lag1'],
        'casos_lag2': ultima_fila['casos_lag1'],
        'incidencia_100k_lag2': ultima_fila['incidencia_100k_lag1'],
    }
    fila['tendencia_incidencia'] = fila['incidencia_100k_lag1'] - fila['incidencia_100k_lag2']

    num_trim = trimestre_a_predecir.quarter
    fila['trimestre_sin'] = np.sin(2 * np.pi * num_trim / 4)
    fila['trimestre_cos'] = np.cos(2 * np.pi * num_trim / 4)

    filas_prediccion.append(fila)

df_prediccion = pd.DataFrame(filas_prediccion)
print("\nEntidades a predecir:", len(df_prediccion))
df_prediccion.head()

Ultimo trimestre con datos: 2026Q2
Trimestre a predecir: 2026Q3

Entidades a predecir: 29


,entidad,trimestre_predicho,temp_max_prom_lag1,temp_min_prom_lag1,lluvia_total_lag1,evapotrans_prom_lag1,casos_lag1,incidencia_100k_lag1,temp_max_prom_lag2,temp_min_prom_lag2,lluvia_total_lag2,evapotrans_prom_lag2,casos_lag2,incidencia_100k_lag2,tendencia_incidencia,trimestre_sin,trimestre_cos
0,AGUASCALIENTES,2026Q3,28.948352,15.358242,186.0,6.007582,37.0,2.360358,24.941111,9.234444,16.8,4.820111,23.0,1.467249,0.893108,-1.0,-1.836970e-16
1,BAJA CALIFORNIA,2026Q3,36.681319,21.069231,0.8,7.865714,26.0,0.620631,28.543333,14.233333,5.0,4.297444,17.0,0.405797,0.214834,-1.0,-1.836970e-16
2,BAJA CALIFORNIA SUR,2026Q3,34.191209,20.142857,5.0,7.400879,340.0,36.886998,28.234444,17.546667,17.6,4.888556,483.0,52.401235,-15.514237,-1.0,-1.836970e-16
3,CAMPECHE,2026Q3,34.318681,24.409890,247.6,5.788022,287.0,29.833246,28.828889,19.386667,56.3,4.242667,263.0,27.338479,2.494766,-1.0,-1.836970e-16
4,CHIAPAS,2026Q3,35.021978,22.215385,316.9,5.436923,711.0,11.494682,30.835556,17.406667,14.7,4.793444,690.0,11.155177,0.339505,-1.0,-1.836970e-16


In [12]:
X_pred = df_prediccion[features]
df_prediccion['riesgo_predicho'] = modelo.predict(X_pred)

proba = modelo.predict_proba(X_pred)
for i, clase in enumerate(modelo.classes_):
    df_prediccion[f'probabilidad_{clase.lower()}'] = proba[:, i]

df_prediccion = df_prediccion.sort_values('riesgo_predicho', key=lambda s: s.map({'ALTO': 0, 'MEDIO': 1, 'BAJO': 2}))

print(df_prediccion[['entidad', 'trimestre_predicho', 'riesgo_predicho',
                      'probabilidad_alto', 'probabilidad_medio', 'probabilidad_bajo']].to_string(index=False))

            entidad trimestre_predicho riesgo_predicho  probabilidad_alto  probabilidad_medio  probabilidad_bajo
           CAMPECHE             2026Q3            ALTO           0.808363            0.189184           0.002453
BAJA CALIFORNIA SUR             2026Q3            ALTO           0.877721            0.119835           0.002444
             COLIMA             2026Q3            ALTO           0.754284            0.239588           0.006128
            CHIAPAS             2026Q3            ALTO           0.553477            0.441439           0.005084
            JALISCO             2026Q3            ALTO           0.815775            0.180978           0.003246
            MORELOS             2026Q3            ALTO           0.872904            0.125460           0.001636
            NAYARIT             2026Q3            ALTO           0.860475            0.137513           0.002012
           GUERRERO             2026Q3            ALTO           0.753253            0.242565   

### 8.2 Agregar `id_geografia` y exportar a CSV para Power BI

Se conecta cada entidad predicha con su `id_geografia` (tomando la fila de
`DIM_GEOGRAFIA` que representa el estado completo, no un municipio específico —
la del municipio "NO ESPECIFICADO" de cada entidad) para que el CSV se pueda
relacionar directamente con `DIM_GEOGRAFIA` en el modelo de Power BI.

In [14]:
dim_geo = pd.read_csv("DIMENSIONES/DIM_GEOGRAFIA.csv")

# id_geografia representativo de cada estado: la fila "NO ESPECIFICADO" (grano estatal)
geo_estatal = dim_geo[dim_geo['municipio'] == 'NO ESPECIFICADO'][['id_geografia', 'entidad']]

resultado_final = df_prediccion.merge(geo_estatal, on='entidad', how='left')
print("Entidades sin id_geografia estatal encontrado:", resultado_final['id_geografia'].isnull().sum())

columnas_export = ['id_geografia', 'entidad', 'trimestre_predicho', 'riesgo_predicho',
                    'probabilidad_alto', 'probabilidad_medio', 'probabilidad_bajo']
resultado_final = resultado_final[columnas_export]

resultado_final.to_csv("PREDICCION_RIESGO_DENGUE.csv", index=False, encoding='utf-8')
print("\nGuardado: PREDICCION_RIESGO_DENGUE.csv")
resultado_final.head(10)

Entidades sin id_geografia estatal encontrado: 0

Guardado: PREDICCION_RIESGO_DENGUE.csv


,id_geografia,entidad,trimestre_predicho,riesgo_predicho,probabilidad_alto,probabilidad_medio,probabilidad_bajo
0,36,CAMPECHE,2026Q3,ALTO,0.808363,0.189184,0.002453
1,24,BAJA CALIFORNIA SUR,2026Q3,ALTO,0.877721,0.119835,0.002444
2,86,COLIMA,2026Q3,ALTO,0.754284,0.239588,0.006128
3,210,CHIAPAS,2026Q3,ALTO,0.553477,0.441439,0.005084
4,675,JALISCO,2026Q3,ALTO,0.815775,0.180978,0.003246
5,951,MORELOS,2026Q3,ALTO,0.872904,0.125460,0.001636
6,972,NAYARIT,2026Q3,ALTO,0.860475,0.137513,0.002012
7,464,GUERRERO,2026Q3,ALTO,0.753253,0.242565,0.004183
8,1844,QUINTANA ROO,2026Q3,ALTO,0.952554,0.046575,0.000871
9,1922,SINALOA,2026Q3,ALTO,0.912554,0.086019,0.001427


### 8.3 Validación contra un caso ya conocido

Se toma una fila real del conjunto de prueba (con su riesgo verdadero ya
calculado) y se compara contra lo que el modelo predice, para verificar el
comportamiento del modelo sobre datos reales en vez de valores inventados.

In [15]:
df_modelo = pd.read_pickle('df_modelo_final.pkl')

# Elegir un estado y trimestre ya ocurridos
fila_prueba = df_modelo[
    (df_modelo['entidad'] == 'SONORA') & (df_modelo['trimestre'] == pd.Period('2026Q2'))
]

X_prueba = fila_prueba[features]
pred = modelo.predict(X_prueba)
proba = modelo.predict_proba(X_prueba)
real = fila_prueba['riesgo'].values

print("Predicción:", pred[0])
print("Real:       ", real[0])
print("Probabilidad por clase:", dict(zip(modelo.classes_, proba[0])))

Predicción: ALTO
Real:        ALTO
Probabilidad por clase: {'ALTO': np.float64(0.9673715613019463), 'BAJO': np.float64(0.0004057197854571883), 'MEDIO': np.float64(0.03222271891259632)}


### 8.4 Validar varios estados a la vez

Útil para revisar de un vistazo qué tan bien predice el modelo sobre un
trimestre completo (todos los estados), comparando predicción vs. valor real.

In [16]:
trimestre_a_validar = pd.Period('2026Q2')

bloque = df_modelo[df_modelo['trimestre'] == trimestre_a_validar].copy()
bloque['prediccion'] = modelo.predict(bloque[features])
bloque['acierto'] = bloque['prediccion'] == bloque['riesgo']

resultado = bloque[['entidad', 'riesgo', 'prediccion', 'acierto']].sort_values('entidad')
print(resultado.to_string(index=False))
print(f"\nAciertos: {bloque['acierto'].sum()} / {len(bloque)} ({bloque['acierto'].mean()*100:.1f}%)")

            entidad riesgo prediccion  acierto
     AGUASCALIENTES  MEDIO      MEDIO     True
    BAJA CALIFORNIA   BAJO      MEDIO    False
BAJA CALIFORNIA SUR   ALTO       ALTO     True
           CAMPECHE   ALTO       ALTO     True
            CHIAPAS  MEDIO       ALTO    False
          CHIHUAHUA   BAJO       BAJO     True
   CIUDAD DE MÉXICO   BAJO       BAJO     True
             COLIMA   ALTO       ALTO     True
            DURANGO  MEDIO      MEDIO     True
         GUANAJUATO  MEDIO      MEDIO     True
           GUERRERO  MEDIO       ALTO    False
            HIDALGO  MEDIO      MEDIO     True
            JALISCO   ALTO       ALTO     True
            MORELOS   ALTO       ALTO     True
             MÉXICO   BAJO      MEDIO    False
            NAYARIT   ALTO       ALTO     True
         NUEVO LEÓN  MEDIO      MEDIO     True
             OAXACA  MEDIO      MEDIO     True
             PUEBLA  MEDIO      MEDIO     True
          QUERÉTARO  MEDIO      MEDIO     True
       QUINTA

## 9. Visualizaciones: matriz de confusión e importancia de variables

Se generan las mismas gráficas incluidas en el reporte final, para tener el
proceso completo documentado en un solo lugar.

In [17]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

plt.rcParams.update({'font.size': 11, 'figure.dpi': 150})

### 9.1 Matriz de confusión

In [18]:
labels = ['ALTO', 'MEDIO', 'BAJO']
cm = confusion_matrix(y_test, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(3)); ax.set_xticklabels(labels)
ax.set_yticks(range(3)); ax.set_yticklabels(labels)
ax.set_xlabel('Predicción'); ax.set_ylabel('Valor real')
ax.set_title(f'Matriz de confusión (prueba: {df_modelo[test_mask]["trimestre"].min()} - {df_modelo[test_mask]["trimestre"].max()})')
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha='center', va='center', color='black', fontsize=12)
plt.tight_layout()
plt.savefig('matriz_confusion.png')
plt.show()

C:\Users\basan\AppData\Local\Temp\ipykernel_5704\4235595539.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 9.2 Importancia de variables

In [19]:
importancia_ordenada = importancia.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(importancia_ordenada.index, importancia_ordenada.values, color='#2E6F8E')
ax.set_title('Importancia de variables en el modelo Random Forest')
ax.set_xlabel('Importancia relativa')
plt.tight_layout()
plt.savefig('importancia_variables.png')
plt.show()

C:\Users\basan\AppData\Local\Temp\ipykernel_5704\604927659.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
